# Behavior experiment (OLMo-1B, base model) — free-form generation + guardrail labeling

Same experiment as `behavior_eval_gemma-2-2b-it.ipynb`, but with **`allenai/OLMo-1B-hf`**, a *base* (non-instruction-tuned) model. We feed each raw statement from `mixed_dataset.csv` with a yes/no framing as a **raw completion** (no chat template) to see how a base model continues the text. Then we eyeball generations, extract the stance with a regex, and label safety with three guardrail models.

> **Run on a GPU host.** Generation uses vLLM; guardrail models load real weights. `shieldgemma-2b` is HF-gated — `huggingface-cli login` or `export HF_TOKEN=...` first.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from latent_alignment.data import load_statements
from latent_alignment.generate import DEFAULT_TEMPLATE, build_prompts, generate_continuations
from latent_alignment.guardrails import GUARDRAILS, extract_stance

In [ ]:
DATASET = "../data/polarity_probing/raw/mixed_dataset.csv"
GEN_MODEL = "allenai/OLMo-1B-hf"
USE_CHAT_TEMPLATE = False  # OLMo-1B is a base model -> raw completion
SAMPLE_N = 30             # small slice first; set to None for the full dataset

df = load_statements(DATASET)
if SAMPLE_N:
    df = df.groupby("label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), SAMPLE_N // 2), random_state=0)
    ).reset_index(drop=True)
print(len(df), "statements")
df.head()

## 1. Generate continuations (vLLM, raw completion)

In [ ]:
statements = df["statement"].tolist()
df["prompt"] = build_prompts(statements, template=DEFAULT_TEMPLATE)
df["generation"] = generate_continuations(
    statements,
    model_name=GEN_MODEL,
    template=DEFAULT_TEMPLATE,
    use_chat_template=USE_CHAT_TEMPLATE,
    max_new_tokens=128,
    temperature=0.7,
)
df[["statement", "label", "generation"]].head(10)

## 2. Eyeball the generations

In [ ]:
with pd.option_context("display.max_colwidth", None):
    for _, row in df.head(20).iterrows():
        print(f"[label={row['label']}] {row['statement']}")
        print(f"  -> {row['generation'].strip()}\n")

## 3. Stance via regex

In [ ]:
df["stance"] = df["generation"].map(extract_stance)
pd.crosstab(df["label"], df["stance"])

## 4. Guardrail labeling (3 models)

In [ ]:
prompts = df["statement"].tolist()
generations = df["generation"].tolist()

# Each classify_* loads its model fresh, so the previous guardrail is freed as we go.
for key, classify in GUARDRAILS.items():
    verdicts = classify(generations, prompts=prompts)
    df[f"{key}_unsafe"] = [v["unsafe"] for v in verdicts]
    df[f"{key}_score"] = [v["score"] for v in verdicts]

## 5. Summary

In [ ]:
unsafe_cols = [c for c in df.columns if c.endswith("_unsafe")]
summary = df.groupby("label")[unsafe_cols].mean()
summary

In [ ]:
df["n_unsafe_votes"] = df[unsafe_cols].sum(axis=1)
print(df["n_unsafe_votes"].value_counts().sort_index())
disagreements = df[df["n_unsafe_votes"].between(1, len(unsafe_cols) - 1)]
disagreements[["statement", "label", "stance", *unsafe_cols]]

In [ ]:
out = Path("../runs/behavior_olmo_1b")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "behavior_results.csv", index=False)
print("wrote", out / "behavior_results.csv")